# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant, if not already installed
!pip install -q mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}, Published: {metadata.date_published}")

## 2. Data Overview
Review all available record sets and their fields. **Entities are referenced by their `@id`s.**

In [ ]:
# List available record sets by their @id and title
record_sets_info = []
for rs in metadata.record_sets:
    info = {
        '@id': rs.id,
        'name': rs.name,
        'description': getattr(rs, 'description', '')
    }
    record_sets_info.append(info)
record_sets_info_df = pd.DataFrame(record_sets_info)
display(record_sets_info_df)

# Show available fields for each record set by @id
for rs in metadata.record_sets:
    print(f"RecordSet @id: {rs.id}, Name: {rs.name}")
    for fld in rs.fields:
        print(f"  Field @id: {fld.id}, Name: {fld.name}, Data type: {fld.data_type}")
    print('-'*40)

For example, let's preview a few records from the main tabular record set.

**Note:** Replace `<record_set_id>` with an actual `@id` from the overview above. (Below, we select the main record set automatically if present.)

In [ ]:
# List a sample of records from the primary record set
# We'll pick the first record set with tabular data (CSV or Excel)
main_rs = None
for rs in metadata.record_sets:
    if getattr(rs, 'data_type', None) in ('csv', 'tsv', 'excel', 'application/csv', 'application/vnd.ms-excel', 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet') or hasattr(rs, 'fields'):
        main_rs = rs
        break

if main_rs is not None:
    print(f"Previewing records from main RecordSet: {main_rs.name} (@id: {main_rs.id})")
    gen = dataset.records(record_set=main_rs.id)
    count = 0
    for rec in gen:
        print(rec)
        count += 1
        if count >= 3:
            break
else:
    print('No tabular record set found.')

## 3. Data Extraction
Load data for one or more record sets into Pandas DataFrames for analysis.

We'll use the `@id`s of the discovered record sets and fields for extraction. All entities are referenced by their `@id`.

In [ ]:
# Gather all record set @ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, use the first DataFrame
demo_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        demo_record_set_id = rsid
        break

if demo_record_set_id:
    print(f"Available columns (field @ids) in '{demo_record_set_id}':")
    print(dataframes[demo_record_set_id].columns.tolist())
    display(dataframes[demo_record_set_id].head())
else:
    print('No DataFrame available for preview.')

## 4. Exploratory Data Analysis (EDA)
Perform common data wrangling steps, such as filtering by a numeric field, normalizing its values, and optionally grouping by another field.

**All field references in code use their `@id` values from the data overview above.**

In [ ]:
import numpy as np

# Choose a numeric field for analysis (by inspecting columns of the demo DataFrame)
numeric_field_id = None
possible_numeric_fields = [col for col in dataframes[demo_record_set_id].columns if dataframes[demo_record_set_id][col].dtype in [np.int64, np.float64, np.int32, np.float32]]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # Try to infer numeric fields (columns whose values look like numbers)
    for col in dataframes[demo_record_set_id].columns:
        try:
            data = pd.to_numeric(dataframes[demo_record_set_id][col], errors='coerce')
            # If enough non-NaN, treat as numeric
            if data.notna().sum() > 0.7*len(data):
                numeric_field_id = col
                dataframes[demo_record_set_id][col] = data
                break
        except Exception:
            continue

if numeric_field_id is None:
    print('No obvious numeric field found for analysis.')
else:
    print(f"Using numeric field @id: '{numeric_field_id}'")
    threshold = dataframes[demo_record_set_id][numeric_field_id].mean()

    filtered_df = dataframes[demo_record_set_id][
        dataframes[demo_record_set_id][numeric_field_id] > threshold
    ].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Find a possible grouping field (categorical)
    group_field = None
    for col in dataframes[demo_record_set_id].columns:
        if col == numeric_field_id:
            continue
        unique_vals = dataframes[demo_record_set_id][col].nunique()
        if 2 < unique_vals < 10:
            group_field = col
            break
    if group_field:
        print(f"Grouping filtered data by field @id: '{group_field}'")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())

## 5. Visualization
Visualize key distributions or relationships for fields in the main record set (referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[demo_record_set_id][numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot of numeric field grouped by a categorical variable
if group_field:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[demo_record_set_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using the [mlcroissant](https://mlcommons.org/croissant/) standard, explored record sets and fields by their `@id`, and performed basic data extraction and EDA using dynamically discovered field IDs and types.

**Key Takeaways:**
- Use entity `@id` fields for all processing and referencing.
- The `mlcroissant` library enables convenient loading and inspection of FAIR datasets.
- The FAIR^2 package covers detailed clinicopathological data on secondary primary colorectal cancer cases in survivors, supporting further analysis and model development.

You can extend EDA and modeling using the provided DataFrames and Croissant semantic metadata.